In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"]="4"

# Flow

We input images and masks. The mask and image size must match, but between images they can have different sizes.

> We want to know which patches correspond to the mask area. So do we resize both the image and the mask and overlap the patches?
> Answer: Yes. This is what we do in the original. So might as well during preprocessing already compute the resized mask and return tensors that have the mask as well as original shape bounding box information. These will be sized for the entire resized image and so will be larger than required.

Compute the valid size for the image, resize and pad the image and mask in parallel. Batch-wise compute mask overlaps using the area interpolate method for all segmentations at once.

Batch-wise compute the patch bounding boxes in the original image sizes. We can first compute the patch bounding boxes for the resized images and then scale based on the scale parameters from the resize.

Finally, for each we need to create the valid mask.

All of these should be 1D arrays that correspond to the shape of the output from DINOv3.

In [3]:
import torch
import gc

# Preprocessing

## Preprocess Images

In [4]:
from aidan_lib.models.dino_lib_compiled import preprocess_imgs, get_compiled_dino, get_onnx_dino, get_normal_dino, get_dino_from_repo, get_dino_from_safetensors, get_compiled_dino_from_repo

Unable to import quantization op. Please install modelopt library (https://github.com/NVIDIA/TensorRT-Model-Optimizer?tab=readme-ov-file#installation) to add support for compiling quantized models


In [5]:
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"PyTorch CUDA Version: {torch.version.cuda}")
print(f"Device Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

CUDA Available: True
PyTorch CUDA Version: 12.4
Device Name: NVIDIA H100 80GB HBM3


In [6]:
device = "cuda"
batch_size = 16
input_size = 1024
dummy_input = torch.randn(batch_size, 3, input_size, input_size, dtype=torch.float16, device=device)

import time
from tqdm import tqdm
def evaluate_model(model, batch_size, iterations=100, warmup_iters=10):
    time_in_model = 0
    with torch.autocast(device_type=device, dtype=torch.bfloat16):
        for i in tqdm(range(iterations + warmup_iters)):
            dummy_input = torch.randn(batch_size, 3, input_size, input_size, dtype=torch.bfloat16, device=device)
            start = time.perf_counter()
            model(dummy_input)
            if i > warmup_iters:
                time_in_model += time.perf_counter() - start

    print(f"Time per iteration: {time_in_model / iterations:0.3f}s")
    return time_in_model / iterations


In [7]:
# repo_dino = get_dino_from_repo(device="cuda")

In [8]:
# evaluate_model(repo_dino.forward_features, iterations=1000)

In [9]:
compiled_repo_dino = get_compiled_dino_from_repo(device=device, backend="inductor", warmup_batch_size=batch_size, max_side_len=input_size, warmup=False, dynamic=False)

Loading dino model
Compiling dino model (backend: inductor, dynamic: False)


In [10]:
# compiled_repo_dino(dummy_input)

In [11]:
test_batch_sizes = [1, 2, 4, 8, 16, 24, 32, 48, 64]

In [12]:
images_per_second = []
for batch_size in test_batch_sizes:
    print(f"Testing with {batch_size}")
    seconds_per_iteration = evaluate_model(compiled_repo_dino, batch_size=batch_size, iterations=100, warmup_iters=20)
    images_per_second.append(batch_size / seconds_per_iteration)
    print(f"Got {images_per_second[-1]} images per second")

Testing with 1


100%|██████████| 120/120 [00:09<00:00, 12.54it/s]


Time per iteration: 0.004s
Got 260.4369755086817 images per second
Testing with 2


100%|██████████| 120/120 [00:08<00:00, 14.38it/s]


Time per iteration: 0.005s
Got 377.35376900864514 images per second
Testing with 4


...

Time per iteration: 0.062s
Got 512.8144573825805 images per second
Testing with 48


100%|██████████| 120/120 [00:28<00:00,  4.22it/s]


Time per iteration: 0.093s
Got 513.6862543715221 images per second
Testing with 64


  0%|          | 0/120 [00:00<?, ?it/s]W0514 14:53:13.068000 1797398 torch/_dynamo/convert_frame.py:844] [0/8] torch._dynamo hit config.cache_size_limit (8)
W0514 14:53:13.068000 1797398 torch/_dynamo/convert_frame.py:844] [0/8]    function: 'forward_features' (/scratch4/home/adempst/projects/aidan-lib/src/aidan_lib/dinov3_repo/dinov3/models/vision_transformer.py:263)
W0514 14:53:13.068000 1797398 torch/_dynamo/convert_frame.py:844] [0/8]    last reason: 0/0: tensor 'L['x']' size mismatch at index 0. expected 1, actual 64
W0514 14:53:13.068000 1797398 torch/_dynamo/convert_frame.py:844] [0/8] To log all recompilation reasons, use TORCH_LOGS="recompiles".
W0514 14:53:13.068000 1797398 torch/_dynamo/convert_frame.py:844] [0/8] To diagnose recompilation issues, see https://pytorch.org/docs/main/torch.compiler_troubleshooting.html.
100%|██████████| 120/120 [00:25<00:00,  4.62it/s]

Time per iteration: 0.213s
Got 300.28828173564557 images per second


In [13]:
del compiled_repo_dino
gc.collect()
torch.cuda.empty_cache()

In [14]:
normal_dino = get_normal_dino(device=device)

Loading facebook/dinov3-vits16-pretrain-lvd1689m...


Loading weights: 100%|██████████| 211/211 [00:00<00:00, 12143.06it/s]


In [15]:
# evaluate_model(normal_dino, iterations=10)
images_per_second = []
for batch_size in test_batch_sizes:
    print(f"Testing with {batch_size}")
    seconds_per_iteration = evaluate_model(normal_dino, batch_size=batch_size, iterations=100, warmup_iters=20)
    images_per_second.append(batch_size / seconds_per_iteration)
    print(f"Got {images_per_second[-1]} images per second")

Testing with 1


100%|██████████| 120/120 [00:00<00:00, 127.19it/s]


Time per iteration: 0.008s
Got 132.81552038215838 images per second
Testing with 2


100%|██████████| 120/120 [00:01<00:00, 109.82it/s]


Time per iteration: 0.009s
Got 225.2109499257573 images per second
Testing with 4


...

Time per iteration: 0.108s
Got 296.99154296374877 images per second
Testing with 48


100%|██████████| 120/120 [00:19<00:00,  6.16it/s]


Time per iteration: 0.161s
Got 298.47626812601413 images per second
Testing with 64


100%|██████████| 120/120 [00:26<00:00,  4.61it/s]

Time per iteration: 0.214s
Got 299.0734453175332 images per second


In [16]:
del normal_dino
gc.collect()
torch.cuda.empty_cache()